# Visual Table Assistant — Training

## Purpose

This notebook is **only for training**. It does not prepare datasets from
scratch.

Flow:

1. Pull the packaged dataset ZIP `datasets/table_assistant_yolo_package.zip`
   from the DVC remote.
2. Extract the ZIP and restore the flat YOLO layout under
   `datasets/table_assistant_yolo/{images,labels}/` using symlink
3. Generate `reports/dataset_splits/{train,val,test}.txt` deterministically
   in this Colab runtime (rarest-class-per-image stratification, seed 42,
   ratios 60/15/25).
4. Validate dataset integrity and split files.
5. Configure MLflow.
6. Run a smoke training pass with YOLO.

## Prerequisites

Before running this notebook, make sure that:

1. **Dataset package is tracked with DVC**: `01_dataset_prep_colab.ipynb` was run, the resulting `datasets/table_assistant_yolo_package.zip` was added to DVC locally (`dvc add` + `dvc push`), and `datasets/table_assistant_yolo_package.zip.dvc` was committed and pushed to git.
2. **Colab Secrets are configured**. In the left sidebar of Colab open the key icon and add:

   - `GDRIVE_CREDENTIALS_DATA`

   This secret must contain the full JSON content of the cached DVC Google Drive credentials generated from a successful local authentication.
3. The Google account used to mount Drive in this session has access to the DVC remote folder.

## 1. Repository setup

Clone the repo (or pull the latest changes if it already exists from a previous
session) into `/content/iaa-visual-table-assistant`.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/LucasGVallejos/iaa-visual-table-assistant.git"
REPO_DIR = Path("/content/iaa-visual-table-assistant")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/iaa-table-assistant")
MLFLOW_DIR = DRIVE_PROJECT_DIR / "mlflow"
YOLO_OUTPUTS_DIR = DRIVE_PROJECT_DIR / "training_outputs"

In [ ]:
%cd /content

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already present, pulling latest changes...")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

## 2. Dependency installation

Install the project's pip requirements. `ultralytics`, `mlflow` and `dvc[gdrive]`
are all declared there.

In [ ]:
!pip install -q -r requirements.txt

## 3. Google Drive mount

Mount Drive so MLflow runs and DVC credentials/cache can persist across Colab
sessions. The DVC remote also resolves through this same mount when running
OAuth authentication.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
YOLO_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"MLFLOW_DIR:       {MLFLOW_DIR}")
print(f"YOLO_OUTPUTS_DIR: {YOLO_OUTPUTS_DIR}")

## 4. DVC pull from the Google Drive remote

This notebook pulls the packaged dataset ZIP from the DVC remote.

To avoid the interactive Google OAuth flow in Colab, DVC credentials are loaded from a Colab Secret named `GDRIVE_CREDENTIALS_DATA`. This secret must contain the JSON credentials previously generated by a successful local DVC authentication.

The notebook pulls only:

`datasets/table_assistant_yolo_package.zip.dvc`

and restores:

`datasets/table_assistant_yolo_package.zip`

In [ ]:
from google.colab import userdata
import os

gdrive_credentials = userdata.get("GDRIVE_CREDENTIALS_DATA")

if not gdrive_credentials:
    raise RuntimeError(
        "Missing Colab Secret: GDRIVE_CREDENTIALS_DATA. "
        "Create it from the cached DVC Google Drive credentials JSON."
    )

os.environ["GDRIVE_CREDENTIALS_DATA"] = gdrive_credentials

print("DVC Google Drive credentials loaded from Colab Secret.")

In [ ]:
PACKAGE_DVC = REPO_DIR / "datasets" / "table_assistant_yolo_package.zip.dvc"
PACKAGE_ZIP = REPO_DIR / "datasets" / "table_assistant_yolo_package.zip"

assert PACKAGE_DVC.exists(), f"Missing DVC file: {PACKAGE_DVC}"

!dvc pull datasets/table_assistant_yolo_package.zip.dvc

assert PACKAGE_ZIP.exists(), f"Missing DVC package zip: {PACKAGE_ZIP}"

size_gb = PACKAGE_ZIP.stat().st_size / (1024 ** 3)
print(f"Found package zip: {PACKAGE_ZIP} ({size_gb:.2f} GB)")

## 5. Extract packaged dataset

The DVC package ZIP is extracted and exposed through the standard dataset path by running the restore script.

After extraction, the package contains:

```text
datasets/table_assistant_yolo_package/
├── table_assistant_yolo/
│   ├── images/
│   └── labels/
└── metadata/
```


In [ ]:
!python -m src.data.preparation.restore_dataset_package

## 6. Generate deterministic split files

This step generates the YOLO split files from the restored flat-layout dataset.

The dataset itself is not physically split into `train/`, `val/` and `test/` folders. Instead, the script writes three text files:

- `reports/dataset_splits/train.txt`
- `reports/dataset_splits/val.txt`
- `reports/dataset_splits/test.txt`

Each file contains absolute image paths for the current Colab runtime. YOLO reads these paths through `configs/data_runtime_colab.yaml` and resolves labels by replacing `images/` with `labels/`.

The split strategy is deterministic:

- train: 60%
- validation: 15%
- test: 25%
- seed: 42
- stratification: rarest class present in each image

In [ ]:
!python -m src.data.preparation.split_dataset

## 7. MLflow setup

Configure MLflow with a tracking URI on Drive so all runs (params, metrics,
artifacts) survive Colab session resets. Ultralytics is told to log to MLflow
via its built-in integration.

In [ ]:
import os
import mlflow
from ultralytics import settings

MLFLOW_EXPERIMENT_NAME = "visual-table-assistant"
MLFLOW_TRACKING_URI = MLFLOW_DIR.as_uri()

MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

settings.update({"mlflow": True})

print("MLflow configured.")
print(f"Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {MLFLOW_EXPERIMENT_NAME}")

## 8. GPU check

Confirm a GPU is attached to this Colab runtime. YOLO will fall back to CPU
if none is available, but training would be unbearably slow, so the cell
warns loudly when CUDA is missing.

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "[WARN] CUDA is not available in this runtime. YOLO will run on CPU "
        "and training will be extremely slow. "
        "Switch to a GPU runtime: Runtime > Change runtime type > GPU."
    )

## 9. Smoke training

Run a short training pass on `yolo26n` to verify the entire pipeline (data_runtime_colab.yaml
→ split files → YOLO loader → training loop → MLflow logging) works end to end.

Five epochs is intentionally short. The goal is to confirm that nothing is
broken before kicking off the real baseline; metrics here are not meaningful.

In [ ]:
import os

from ultralytics import YOLO

os.environ["MLFLOW_RUN"] = "smoke_yolo26n_001"

smoke_model = YOLO("yolo26n.pt")

smoke_results = smoke_model.train(
    data=str(REPO_DIR / "configs" / "data_runtime_colab.yaml"),
    epochs=5,
    imgsz=640,
    batch=16,
    project=str(YOLO_OUTPUTS_DIR),
    name="smoke_yolo26n_001",
    exist_ok=True,
)

print("Smoke training finished.")
print(f"Run dir: {smoke_results.save_dir}")

## 10. Baseline training

First real training pass. Goal: a measurable, reproducible baseline.

**Outline**

1. **Config** — every hyperparameter as a named constant. `MODEL_SIZE` is the only knob to alternate runs across `n`, `s`, `m`, `l`, `x`.
2. **Train** — `epochs=50`, `patience=15`, `seed=42`, `cache='disk'`. The single long-running cell.
3. **Final evaluation** — re-validates with `best.pt` over `val` (sanity at the chosen checkpoint) and over `test` (the 25% holdout that will be reported as the headline metric).
4. **Summary** — JSON with config + per-split metrics + per-class mAP. Saved to `<save_dir>/baseline_summary.json` (Drive, automatic) and mirrored to `reports/baselines/` for committing from your local checkout.

**Deliberate decisions**

- No class weighting / focal loss in this baseline. The dataset has a documented food/knife imbalance ~32:1 and the goal is to measure the uncorrected impact first.
- Model family stays YOLO26 across smoke and baseline so the smoke validates the actual loader path.
- `cache='disk'` consumes ~50 GB of Colab scratch but keeps epochs fast.
- No git push from Colab. The summary lives on Drive automatically; for git, copy the JSON from Drive into the local checkout and commit there.

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime

import torch

# Toggle this to alternate runs across model sizes without touching the rest.
MODEL_SIZE = "m"  # one of: n, s, m, l, x

BASELINE_MODEL    = f"yolo26{MODEL_SIZE}.pt"
BASELINE_RUN_NAME = f"baseline_yolo26{MODEL_SIZE}_001"
DATA_YAML         = REPO_DIR / "configs" / "data_runtime_colab.yaml"

EPOCHS, IMGSZ, BATCH = 30, 640, 16
PATIENCE, SEED       = 7, 42
DEVICE               = 0 if torch.cuda.is_available() else "cpu"

assert DATA_YAML.exists(), f"Missing YOLO data config: {DATA_YAML}"

print(f"Model:      {BASELINE_MODEL}")
print(f"Run name:   {BASELINE_RUN_NAME}")
print(f"Data YAML:  {DATA_YAML}")
print(f"Epochs:     {EPOCHS}  imgsz: {IMGSZ}  batch: {BATCH}")
print(f"Patience:   {PATIENCE}  seed: {SEED}")
print(f"Device:     {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
else:
    print("[WARN] CUDA not available. CPU training will be unusably slow.")

In [ ]:
from ultralytics import YOLO

os.environ["MLFLOW_RUN"] = BASELINE_RUN_NAME

baseline_model = YOLO(BASELINE_MODEL)
baseline_results = baseline_model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    device=DEVICE,
    cache="disk",
    project=str(YOLO_OUTPUTS_DIR),
    name=BASELINE_RUN_NAME,
    exist_ok=True,
)

baseline_save_dir = Path(baseline_results.save_dir)
best_model_path   = baseline_save_dir / "weights" / "best.pt"
last_model_path   = baseline_save_dir / "weights" / "last.pt"

assert best_model_path.exists(), f"Missing best model: {best_model_path}"

print("\nTraining finished.")
print(f"Run dir:    {baseline_save_dir}")
print(f"Best model: {best_model_path}")
print(f"Last model: {last_model_path}")

### 10.1 Final evaluation

Two passes with `best.pt` to produce clean post-training metrics:

- **`val` split**: sanity check at the chosen checkpoint. The numbers should match (within noise) the best epoch's val metrics already logged by training; if they diverge, something is wrong with checkpoint selection.
- **`test` split**: the 25% holdout untouched during training and val. This is the metric to cite as the baseline result.

In [ ]:
best_model = YOLO(str(best_model_path))

val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(YOLO_OUTPUTS_DIR),
    name=f"{BASELINE_RUN_NAME}_val",
    exist_ok=True,
    plots=True,
)

print(f"val mAP50-95: {val_metrics.box.map:.4f}  mAP50: {val_metrics.box.map50:.4f}")

In [ ]:
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(YOLO_OUTPUTS_DIR),
    name=f"{BASELINE_RUN_NAME}_test",
    exist_ok=True,
    plots=True,
)

print(f"test mAP50-95: {test_metrics.box.map:.4f}  mAP50: {test_metrics.box.map50:.4f}")

### 10.2 Summary persistence

The summary captures config + val_metrics + test_metrics, with per-class `mAP@50-95` so the imbalance impact can be diagnosed without re-running anything.

Two writes:

- `<save_dir>/baseline_summary.json` lives under `training_outputs/` on Drive, so it is auto-persisted across Colab sessions.
- `reports/baselines/baseline_yolo26<size>_<YYYYMMDD_HHMMSS>.json` is written into the Colab clone of the repo. Because the clone is ephemeral, the file lands in git only after you copy it from Drive (same JSON) into your local checkout and commit it from there.

In [ ]:
def _safe_float(v):
    try:
        return float(v)
    except Exception:
        return None

def _per_class_map5095(metrics):
    """Per-class mAP@50-95 keyed by class name."""
    box = metrics.box
    out = {}
    # box.maps is a length-nc array indexed by class id; box.ap_class_index lists
    # the class ids that actually appeared in this eval.
    for class_idx in box.ap_class_index:
        cid = int(class_idx)
        out[metrics.names[cid]] = _safe_float(box.maps[cid])
    return out

def _split_block(metrics):
    return {
        "mAP50_95":           _safe_float(metrics.box.map),
        "mAP50":              _safe_float(metrics.box.map50),
        "mAP75":              _safe_float(metrics.box.map75),
        "precision":          _safe_float(metrics.box.mp),
        "recall":             _safe_float(metrics.box.mr),
        "per_class_mAP50_95": _per_class_map5095(metrics),
    }

summary = {
    "created_at":   datetime.now().isoformat(timespec="seconds"),
    "experiment":   MLFLOW_EXPERIMENT_NAME,
    "run_name":     BASELINE_RUN_NAME,
    "model":        BASELINE_MODEL,
    "data_yaml":    str(DATA_YAML),
    "config": {
        "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
        "patience": PATIENCE, "seed": SEED, "device": str(DEVICE),
        "cache": "disk",
    },
    "save_dir":     str(baseline_save_dir),
    "best_model":   str(best_model_path),
    "val_metrics":  _split_block(val_metrics),
    "test_metrics": _split_block(test_metrics),
}

# 1) Drive — auto-persisted via training_outputs/<run_name>/.
drive_summary_path = baseline_save_dir / "baseline_summary.json"
drive_summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

# 2) Repo working tree — manual commit from local checkout afterwards.
local_dir = REPO_DIR / "reports" / "baselines"
local_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
local_summary_path = local_dir / f"baseline_yolo26{MODEL_SIZE}_{stamp}.json"
local_summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Drive: {drive_summary_path}")
print(f"Repo:  {local_summary_path}")
print(f"\ntest mAP50-95 = {summary['test_metrics']['mAP50_95']}")